# TP Análisis de Datos - Ecobici 2024
## Actividad Grupal 2 - Análisis Exploratorio de Datos (EDA)

Dataset: recorridos realizados de Ecobici, Buenos Aires, 2024.

Contenidos según Unidad 2 del programa:
- Estadísticas descriptivas
- Estadística robusta
- Relaciones entre variables
- Visualización de variables numéricas y categóricas


## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PATH = "badata_ecobici_recorridos_realizados_2024.csv"

# El archivo es grande (~800MB). Definimos dtypes para acelerar la carga y bajar el uso de memoria.
dtypes = {
    "id_recorrido": "int64",
    "duracion_recorrido": "float64",
    "id_estacion_origen": "Int64",
    "nombre_estacion_origen": "category",
    "direccion_estacion_origen": "category",
    "long_estacion_origen": "float64",
    "lat_estacion_origen": "float64",
    "id_estacion_destino": "Int64",
    "nombre_estacion_destino": "category",
    "direccion_estacion_destino": "category",
    "long_estacion_destino": "float64",
    "lat_estacion_destino": "float64",
    "id_usuario": "float64",
    "modelo_bicicleta": "category",
    "genero": "category",
}
parse_dates = ["fecha_origen_recorrido", "fecha_destino_recorrido"]

df = pd.read_csv(PATH, dtype=dtypes, parse_dates=parse_dates)
print(df.shape)
df.head()


## 2. Estructura y tipos de variables

In [ ]:
df.info(memory_usage="deep")


In [ ]:
# Clasificación de variables
numericas = ["duracion_recorrido", "long_estacion_origen", "lat_estacion_origen",
             "long_estacion_destino", "lat_estacion_destino"]
categoricas = ["genero", "modelo_bicicleta", "nombre_estacion_origen", "nombre_estacion_destino"]
fechas = ["fecha_origen_recorrido", "fecha_destino_recorrido"]

print("Numéricas:", numericas)
print("Categóricas:", categoricas)
print("Fechas:", fechas)


## 3. Datos faltantes

In [ ]:
na = df.isna().sum().sort_values(ascending=False)
na_pct = (na / len(df) * 100).round(2)
pd.DataFrame({"faltantes": na, "%": na_pct})


## 4. Estadísticas descriptivas - Variables numéricas

In [ ]:
df[numericas].describe().T


### Estadística robusta

La duración del recorrido suele tener outliers (recorridos anormalmente largos por
bicicletas no devueltas, o cortos por errores de estación). Complementamos la media/desvío
(sensibles a outliers) con mediana, IQR y MAD.


In [ ]:
def robust_stats(s):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    mediana = s.median()
    mad = (s - mediana).abs().median()
    return pd.Series({
        "mediana": mediana,
        "IQR": iqr,
        "Q1": q1,
        "Q3": q3,
        "MAD": mad,
        "limite_inferior_outliers": q1 - 1.5 * iqr,
        "limite_superior_outliers": q3 + 1.5 * iqr,
    })

robust_stats(df["duracion_recorrido"])


In [ ]:
# Duración en minutos, y proporción de outliers según regla IQR
dur_min = df["duracion_recorrido"] / 60
q1, q3 = dur_min.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = ((dur_min < lo) | (dur_min > hi)).mean() * 100
print(f"Duración (min) - mediana: {dur_min.median():.1f}, IQR: [{q1:.1f}, {q3:.1f}]")
print(f"% de recorridos fuera de rango (outliers por IQR): {outliers:.2f}%")


## 5. Estadísticas descriptivas - Variables categóricas

In [ ]:
for col in ["genero", "modelo_bicicleta"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print(df[col].value_counts(normalize=True, dropna=False).round(3) * 100)
    print()


In [ ]:
print("Estaciones de origen más usadas:")
df["nombre_estacion_origen"].value_counts().head(10)


## 6. Visualización de variables numéricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución completa (con outliers) vs recortada (percentil 99)
sns.histplot(dur_min, bins=60, ax=axes[0])
axes[0].set_title("Duración del recorrido (min) - completa")
axes[0].set_xlabel("minutos")

p99 = dur_min.quantile(0.99)
sns.histplot(dur_min[dur_min <= p99], bins=60, ax=axes[1])
axes[1].set_title("Duración del recorrido (min) - hasta P99")
axes[1].set_xlabel("minutos")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x=dur_min[dur_min <= p99])
plt.title("Boxplot duración de recorrido (min, hasta P99)")
plt.xlabel("minutos")
plt.show()


## 7. Visualización de variables categóricas

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="genero", order=df["genero"].value_counts().index)
plt.title("Recorridos por género")
plt.show()


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="modelo_bicicleta", order=df["modelo_bicicleta"].value_counts().index)
plt.title("Recorridos por modelo de bicicleta")
plt.show()


## 8. Relaciones entre variables

### Numérica vs. categórica: duración según género y modelo de bicicleta

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df.assign(dur_min=dur_min), x="genero", y="dur_min", showfliers=False)
plt.title("Duración de recorrido (min) según género (sin outliers)")
plt.ylabel("minutos")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df.assign(dur_min=dur_min), x="modelo_bicicleta", y="dur_min", showfliers=False)
plt.title("Duración de recorrido (min) según modelo de bicicleta (sin outliers)")
plt.ylabel("minutos")
plt.show()


### Categórica vs. categórica: género según modelo de bicicleta

In [ ]:
tab = pd.crosstab(df["modelo_bicicleta"], df["genero"], normalize="index") * 100
tab.round(1)


In [ ]:
tab.plot(kind="bar", stacked=True, figsize=(6, 4))
plt.title("Distribución de género por modelo de bicicleta (%)")
plt.ylabel("%")
plt.legend(title="Género", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


### Variación temporal: recorridos por hora del día y día de la semana

In [ ]:
df["hora"] = df["fecha_origen_recorrido"].dt.hour
df["dia_semana"] = df["fecha_origen_recorrido"].dt.day_name()

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="hora", color="steelblue")
plt.title("Recorridos por hora de inicio")
plt.show()


In [ ]:
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="dia_semana", order=orden_dias, color="steelblue")
plt.title("Recorridos por día de la semana")
plt.xticks(rotation=30)
plt.show()


### Numérica vs. numérica: correlación entre variables numéricas

In [ ]:
corr = df[numericas].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación - variables numéricas")
plt.show()
corr


## 9. Conclusiones preliminares

_(Completar con las observaciones del grupo: outliers en duración, estaciones más usadas,
patrones horarios, diferencias por género/modelo de bicicleta, variables candidatas a target
y features para el problema de ML supervisado a definir.)_
